# DuoDiT FLOPs and VRAM Profiler

This notebook profiles DuoDiT's current x2 fine-tuning architecture using synthetic latent inputs. It applies the same trainable-parameter selection as `train_x2_finetune.py`; checkpoint values do not change FLOPs or parameter VRAM.

The FLOP helper reports both PyTorch's hardware convention and the one-MAC-per-FLOP convention used by the DiT paper. The memory tracker measures actual CUDA peaks for an epoch.

## Setup

Only PyTorch is required.

In [ ]:
from __future__ import annotations

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## Forward FLOPs

Pass a callable that performs exactly one model forward. `batch_size` is used only to normalize the result per sample.

In [ ]:
from collections.abc import Callable
from typing import Any

from torch.profiler import ProfilerActivity, profile


def measure_forward_flops(
    forward_fn: Callable[[], Any],
    batch_size: int = 1,
) -> dict[str, float]:
    """Profile one forward call and return batch and per-sample GFLOPs."""
    if not callable(forward_fn):
        raise TypeError("forward_fn must be a zero-argument callable")
    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")

    has_cuda = torch.cuda.is_available()
    activities = [ProfilerActivity.CPU]
    if has_cuda:
        activities.append(ProfilerActivity.CUDA)
        torch.cuda.synchronize()

    with torch.inference_mode():
        with profile(
            activities=activities,
            record_shapes=True,
            with_flops=True,
            acc_events=True,
        ) as profiler:
            forward_fn()

    if has_cuda:
        torch.cuda.synchronize()

    hardware_flops = float(
        sum(event.flops or 0 for event in profiler.key_averages())
    )
    hardware_gflops_batch = hardware_flops / 1e9
    hardware_gflops_per_sample = hardware_gflops_batch / batch_size

    return {
        "hardware_gflops_batch": hardware_gflops_batch,
        "hardware_gflops_per_sample": hardware_gflops_per_sample,
        # DiT/fvcore convention: one multiply-accumulate counts as one FLOP.
        "paper_gflops_per_sample": hardware_gflops_per_sample / 2,
    }


### CPU-safe example

This verifies the helper without downloading a model or requiring CUDA.

In [ ]:
tiny_model = torch.nn.Sequential(
    torch.nn.Linear(32, 64),
    torch.nn.GELU(),
    torch.nn.Linear(64, 16),
)
tiny_inputs = torch.randn(8, 32)

tiny_stats = measure_forward_flops(
    lambda: tiny_model(tiny_inputs),
    batch_size=tiny_inputs.shape[0],
)
tiny_stats


## DuoDiT configuration

Set `RUN_DUODIT_PROFILE = True` on the RTX 4090. `X2_VIT_DEPTH` matches the training CLI choices. `FORCE_LOCAL_TIMM_VIT` creates the same ViT-L architecture without downloading pretrained weights.

In [ ]:
RUN_DUODIT_PROFILE = False

MODEL = "DiT-XL/2"
IMAGE_SIZE = 256
NUM_CLASSES = 1_000
X2_VIT_DEPTH = 1
TRAINING_MODE = "x2_finetune"  # "x2_finetune" or "full"
FORCE_LOCAL_TIMM_VIT = True

PROFILE_BATCH_SIZE = 1
PROFILE_EPOCHS = 2
PROFILE_STEPS_PER_EPOCH = 2
INCLUDE_EMA = True
INFERENCE_WARMUP_STEPS = 5
INFERENCE_PROFILE_STEPS = 20
GPU_SAMPLE_INTERVAL_SECONDS = 0.05

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE


### Build DuoDiT and select trainable parameters

The x2 mode reproduces the freezing logic in `train_x2_finetune.py`. The local timm override changes only weight initialization, not architecture or compute.

In [ ]:
if RUN_DUODIT_PROFILE:
    if DEVICE.type != "cuda":
        raise RuntimeError("Enable this profile on a CUDA machine")

    import contextlib
    import io

    from diffusion import create_diffusion
    from models import DiT_models

    latent_size = IMAGE_SIZE // 8

    if FORCE_LOCAL_TIMM_VIT:
        import timm

        original_create_model = timm.create_model

        def create_model_without_download(*args, **kwargs):
            kwargs["pretrained"] = False
            return original_create_model(*args, **kwargs)

        timm.create_model = create_model_without_download
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                model = DiT_models[MODEL](
                    input_size=latent_size,
                    num_classes=NUM_CLASSES,
                    x2_vit_depth=X2_VIT_DEPTH,
                )
        finally:
            timm.create_model = original_create_model
    else:
        model = DiT_models[MODEL](
            input_size=latent_size,
            num_classes=NUM_CLASSES,
            x2_vit_depth=X2_VIT_DEPTH,
        )

    if TRAINING_MODE == "x2_finetune":
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.x2_embedder.parameters():
            parameter.requires_grad = True
        model.x2_cls_tokens.requires_grad = True
        for parameter in model.x2_vit_blocks.parameters():
            parameter.requires_grad = True
        if model.x2_vit_proj_in is not None:
            for parameter in model.x2_vit_proj_in.parameters():
                parameter.requires_grad = True
        if model.x2_vit_proj_out is not None:
            for parameter in model.x2_vit_proj_out.parameters():
                parameter.requires_grad = True
        for parameter in model.final_layer.parameters():
            parameter.requires_grad = True
    elif TRAINING_MODE != "full":
        raise ValueError("TRAINING_MODE must be 'x2_finetune' or 'full'")

    model = model.to(DEVICE)
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    total_params = sum(parameter.numel() for parameter in model.parameters())
    latents = torch.randn(
        PROFILE_BATCH_SIZE, 4, latent_size, latent_size, device=DEVICE
    )
    timesteps = torch.randint(
        0, 1_000, (PROFILE_BATCH_SIZE,), device=DEVICE
    )
    labels = torch.randint(
        0, NUM_CLASSES, (PROFILE_BATCH_SIZE,), device=DEVICE
    )

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")


### Measure the DuoDiT forward

In [ ]:
if RUN_DUODIT_PROFILE:
    previous_mode = model.training
    model.eval()

    def quiet_duodit_forward():
        with contextlib.redirect_stdout(io.StringIO()):
            return model(latents, timesteps, labels)

    duodit_flops = measure_forward_flops(
        quiet_duodit_forward,
        batch_size=PROFILE_BATCH_SIZE,
    )
    model.train(previous_mode)

    print(
        f"DiT paper convention: "
        f"{duodit_flops['paper_gflops_per_sample']:.2f} GFLOPs/sample"
    )
    print(
        f"Hardware convention: "
        f"{duodit_flops['hardware_gflops_per_sample']:.2f} GFLOPs/sample"
    )


## Inference GPU usage

This benchmark warms up the model, then measures repeated forward passes. PyTorch reports latency, throughput, and peak tensor/allocator VRAM. A background `nvidia-smi` sampler reports mean and maximum GPU utilization, memory utilization, used VRAM, and power.

In [ ]:
import statistics
import subprocess
import threading
import time


def _parse_nvidia_value(value: str) -> float | None:
    value = value.strip()
    if value in {"", "N/A", "[N/A]"}:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def query_nvidia_smi(gpu_index: int) -> dict[str, float | str] | None:
    fields = (
        "name,utilization.gpu,utilization.memory,"
        "memory.used,memory.total,power.draw"
    )
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                f"--id={gpu_index}",
                f"--query-gpu={fields}",
                "--format=csv,noheader,nounits",
            ],
            check=False,
            capture_output=True,
            text=True,
            timeout=5,
        )
    except (OSError, subprocess.SubprocessError):
        return None

    if result.returncode != 0 or not result.stdout.strip():
        return None

    values = [part.strip() for part in result.stdout.splitlines()[0].split(",")]
    if len(values) != 6:
        return None

    return {
        "gpu_name": values[0],
        "gpu_util_percent": _parse_nvidia_value(values[1]),
        "memory_util_percent": _parse_nvidia_value(values[2]),
        "memory_used_mib": _parse_nvidia_value(values[3]),
        "memory_total_mib": _parse_nvidia_value(values[4]),
        "power_w": _parse_nvidia_value(values[5]),
    }


class NvidiaSmiSampler:
    """Sample driver-level GPU metrics while inference is running."""

    def __init__(self, gpu_index: int, interval_seconds: float = 0.05):
        self.gpu_index = gpu_index
        self.interval_seconds = interval_seconds
        self.samples: list[dict[str, float | str]] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def _run(self) -> None:
        while not self._stop.is_set():
            sample = query_nvidia_smi(self.gpu_index)
            if sample is not None:
                self.samples.append(sample)
            self._stop.wait(self.interval_seconds)

    def start(self) -> None:
        self.samples.clear()
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self) -> list[dict[str, float | str]]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=max(1.0, 2 * self.interval_seconds))
        return list(self.samples)


def _sample_stat(
    samples: list[dict[str, float | str]],
    key: str,
    operation,
) -> float | None:
    values = [sample[key] for sample in samples if sample.get(key) is not None]
    return float(operation(values)) if values else None


def profile_cuda_inference(
    forward_fn: Callable[[], Any],
    batch_size: int,
    device: str | torch.device = "cuda",
    warmup_steps: int = 5,
    profile_steps: int = 20,
    sample_interval_seconds: float = 0.05,
) -> dict[str, float | int | str | None]:
    """Measure inference latency, throughput, VRAM, utilization, and power."""
    cuda_device = torch.device(device)
    if not torch.cuda.is_available() or cuda_device.type != "cuda":
        raise RuntimeError("CUDA is required for inference GPU profiling")
    if batch_size < 1 or warmup_steps < 0 or profile_steps < 1:
        raise ValueError("batch_size/profile_steps must be positive; warmup may be zero")

    gpu_index = cuda_device.index
    if gpu_index is None:
        gpu_index = torch.cuda.current_device()

    with torch.inference_mode():
        for _ in range(warmup_steps):
            forward_fn()
    torch.cuda.synchronize(cuda_device)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(cuda_device)

    sampler = NvidiaSmiSampler(gpu_index, sample_interval_seconds)
    sampler.start()
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            for _ in range(profile_steps):
                forward_fn()
        torch.cuda.synchronize(cuda_device)
    finally:
        elapsed_seconds = time.perf_counter() - started
        samples = sampler.stop()

    total_samples = batch_size * profile_steps
    gib = 2**30
    mean_gpu_util = _sample_stat(samples, "gpu_util_percent", statistics.mean)
    max_gpu_util = _sample_stat(samples, "gpu_util_percent", max)
    mean_memory_util = _sample_stat(samples, "memory_util_percent", statistics.mean)
    max_memory_util = _sample_stat(samples, "memory_util_percent", max)
    mean_memory_used_mib = _sample_stat(samples, "memory_used_mib", statistics.mean)
    max_memory_used_mib = _sample_stat(samples, "memory_used_mib", max)
    mean_power = _sample_stat(samples, "power_w", statistics.mean)
    max_power = _sample_stat(samples, "power_w", max)

    return {
        "gpu_name": samples[0]["gpu_name"] if samples else torch.cuda.get_device_name(cuda_device),
        "batch_size": batch_size,
        "warmup_steps": warmup_steps,
        "profile_steps": profile_steps,
        "latency_ms_per_batch": 1_000 * elapsed_seconds / profile_steps,
        "samples_per_second": total_samples / elapsed_seconds,
        "peak_allocated_gib": torch.cuda.max_memory_allocated(cuda_device) / gib,
        "peak_reserved_gib": torch.cuda.max_memory_reserved(cuda_device) / gib,
        "mean_gpu_util_percent": mean_gpu_util,
        "max_gpu_util_percent": max_gpu_util,
        "mean_memory_util_percent": mean_memory_util,
        "max_memory_util_percent": max_memory_util,
        "mean_driver_memory_used_gib": (
            mean_memory_used_mib / 1_024 if mean_memory_used_mib is not None else None
        ),
        "max_driver_memory_used_gib": (
            max_memory_used_mib / 1_024 if max_memory_used_mib is not None else None
        ),
        "mean_power_w": mean_power,
        "max_power_w": max_power,
        "nvidia_smi_samples": len(samples),
    }

In [ ]:
if RUN_DUODIT_PROFILE:
    previous_mode = model.training
    model.eval()
    inference_gpu = profile_cuda_inference(
        quiet_duodit_forward,
        batch_size=PROFILE_BATCH_SIZE,
        device=DEVICE,
        warmup_steps=INFERENCE_WARMUP_STEPS,
        profile_steps=INFERENCE_PROFILE_STEPS,
        sample_interval_seconds=GPU_SAMPLE_INTERVAL_SECONDS,
    )
    model.train(previous_mode)

    print("Inference GPU usage")
    for name, value in inference_gpu.items():
        if isinstance(value, float):
            print(f"{name}: {value:.3f}")
        else:
            print(f"{name}: {value}")

## CUDA VRAM per epoch

`peak_allocated_gib` is live tensor memory. `peak_reserved_gib` includes the CUDA caching allocator and is normally larger.

In [ ]:
class CudaEpochMemoryTracker:
    """Measure allocated and reserved CUDA memory peaks for one epoch."""

    def __init__(self, device: str | torch.device | None = None):
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is required for VRAM measurement")

        self.device = torch.device(device or "cuda")
        if self.device.type != "cuda":
            raise ValueError("device must be a CUDA device")

        self._started = False
        self.stats: dict[str, float] | None = None

    def start_epoch(self) -> None:
        torch.cuda.synchronize(self.device)
        torch.cuda.reset_peak_memory_stats(self.device)
        self._baseline_bytes = torch.cuda.memory_allocated(self.device)
        self._started = True
        self.stats = None

    def end_epoch(self) -> dict[str, float]:
        if not self._started:
            raise RuntimeError("start_epoch() must be called first")

        torch.cuda.synchronize(self.device)
        gib = 2**30
        peak_allocated = torch.cuda.max_memory_allocated(self.device)
        self.stats = {
            "baseline_allocated_gib": self._baseline_bytes / gib,
            "peak_allocated_gib": peak_allocated / gib,
            "epoch_peak_delta_gib": max(0, peak_allocated - self._baseline_bytes) / gib,
            "peak_reserved_gib": torch.cuda.max_memory_reserved(self.device) / gib,
        }
        self._started = False
        return self.stats

    def __enter__(self) -> "CudaEpochMemoryTracker":
        self.start_epoch()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.end_epoch()
        return False


### Synthetic DuoDiT training epochs

This runs the same diffusion loss, backward pass, and AdamW update as the training scripts. `INCLUDE_EMA` allocates their full EMA model copy. Synthetic latents omit VAE encoding and DDP buffers; use the final integration cell for the complete real epoch.

In [ ]:
if RUN_DUODIT_PROFILE:
    from copy import deepcopy

    diffusion = create_diffusion(timestep_respacing="")
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=1e-4,
        weight_decay=0,
    )
    ema = deepcopy(model).eval() if INCLUDE_EMA else None
    if ema is not None:
        ema.requires_grad_(False)

    tracker = CudaEpochMemoryTracker(DEVICE)
    model.train()

    for epoch in range(PROFILE_EPOCHS):
        tracker.start_epoch()
        for _ in range(PROFILE_STEPS_PER_EPOCH):
            timesteps = torch.randint(
                0, diffusion.num_timesteps, (PROFILE_BATCH_SIZE,), device=DEVICE
            )
            labels = torch.randint(
                0, NUM_CLASSES, (PROFILE_BATCH_SIZE,), device=DEVICE
            )
            with contextlib.redirect_stdout(io.StringIO()):
                loss = diffusion.training_losses(
                    model, latents, timesteps, {"y": labels}
                )["loss"].mean()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        memory = tracker.end_epoch()
        print(
            f"epoch={epoch} loss={loss.item():.4f} "
            f"allocated={memory['peak_allocated_gib']:.2f} GiB "
            f"reserved={memory['peak_reserved_gib']:.2f} GiB"
        )


### Measure the complete `train_x2_finetune.py` epoch

For the final RTX 4090 value, place these calls around the existing inner training loop. This includes the actual VAE, DDP buckets, optimizer, EMA, and allocator behavior.

In [ ]:
# Add once before `for epoch in range(args.epochs)`:
# tracker = CudaEpochMemoryTracker(device)

# Add at the beginning of each epoch:
# tracker.start_epoch()

# Add after the inner `for x, y in loader` loop:
# memory = tracker.end_epoch()
# logger.info(
#     f"Epoch {epoch}: "
#     f"peak allocated={memory['peak_allocated_gib']:.2f} GiB, "
#     f"peak reserved={memory['peak_reserved_gib']:.2f} GiB"
# )


## Interpretation

- Change x2 ViT depth, training mode, image size, or batch size in the configuration cell and rerun the model cells.
- `x2_finetune` reports the exact trainable subset selected by `train_x2_finetune.py`; `full` leaves every model parameter trainable.
- PyTorch estimates FLOPs for supported operators, primarily matrix multiplication and convolution. Custom fused kernels may be omitted.
- The DiT paper number is approximately half the PyTorch hardware count because it treats one multiply-accumulate as one FLOP.
- CUDA peaks depend on precision, optimizer state, activation checkpointing, allocator fragmentation, and software versions. Measure on the target GPU.
- FLOPs are normally constant across epochs; VRAM should also be stable unless shapes or training behavior change.